In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="st_all_distilroberta_v1_cosine_threshold_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
model_name = "sentence-transformers/all-distilroberta-v1"

model = SentenceTransformer(model_name, device=str(device))
print(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/all-distilroberta-v1


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
batch_size = 128

emb1 = model.encode(
    sent1,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

emb2 = model.encode(
    sent2,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 768)
emb2 shape: (408, 768)


In [ ]:
emb1_list = emb1.cpu().float().tolist()
emb2_list = emb2.cpu().float().tolist()
vault.create_embedding_list("sentence-transformers-sentence-1-threshold_2", ndim=768)
vault.create_embedding_list("sentence-transformers-sentence-2-threshold_2", ndim=768)

for i in range(len(emb1_list)):
    vault.append_embedding("sentence-transformers-sentence-1-threshold_2", emb1_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )
    vault.append_embedding("sentence-transformers-sentence-2-threshold_2", emb2_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )


description = "This dataset stores the sentence embeddings for the sentence1 field of the GLUE MRPC validation set. Each item is a 768-dimensional float vector produced by the sentence-transformers/all-distilroberta-v1 model with normalized embeddings enabled, so dot products correspond to cosine similarity. The dataset is structured as an embedding list: one embedding per MRPC validation example, with each embedding linked to the corresponding row in glue_mrpc_validation through input item provenance. In this workflow, it provides the first-sentence representation used with the matching sentence-transformers-sentence-2-threshold_2 embeddings to compute similarity scores and apply a 0.72 threshold for paraphrase prediction."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-1-threshold_2", description, embedding)

properties = {"task": "paraphrase detection", "representation": "text embedding", "text_field": "sentence1", "model": "sentence-transformers/all-distilroberta-v1", "embedding_dim": "768", "normalized": "true", "similarity": "cosine", "split": "validation", "size": "408", "source": "glue/mrpc", "dataset_name": "sentence-transformers-sentence-1-threshold_2"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-1-threshold_2", cat, embedding, prop)

description = "This dataset stores the sentence embedding for the second sentence (sentence2) of each example in the GLUE MRPC validation set. It is an embedding list with one row per MRPC record, where each item contains a 768-dimensional, L2-normalized vector produced by the sentence-transformers/all-distilroberta-v1 model, and each embedding is linked back to the corresponding source row in glue_mrpc_validation through the input_items index range [i, i+1]. In this workflow, these sentence2 embeddings are paired with the corresponding sentence1 embeddings from sentence-transformers-sentence-1-threshold_2, and their dot product is used as cosine similarity to make threshold-based paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-2-threshold_2", description, embedding)

properties = {"task": "paraphrase detection", "representation": "sentence embedding", "text_field": "sentence2", "embedding_model": "sentence-transformers/all-distilroberta-v1", "similarity": "cosine", "normalized": "true", "embedding_dim": "768", "split": "validation", "size": "408", "source": "glue/mrpc", "modality": "text", "language": "english"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-2-threshold_2", cat, embedding, prop)

In [6]:
scores = torch.sum(emb1 * emb2, dim=1)
threshold = 0.72
y_pred = (scores >= threshold).to(torch.int64).cpu().numpy()
scores_np = scores.cpu().numpy()

print("done")
print("threshold:", threshold)
print("score_range:", (float(scores_np.min()), float(scores_np.max())))

done
threshold: 0.72
score_range: (0.22469396889209747, 0.9964145421981812)


In [ ]:
vault.create_record_list("sentence_transformers_mrpc_prediction_threshold_2", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("sentence_transformers_mrpc_prediction_threshold_2", {"prediction": y_pred[i]}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "sentence-transformers-sentence-1-threshold_2": [i, i + 1],
                           "sentence-transformers-sentence-2-threshold_2": [i, i + 1],
                       }
                       )

description = "Binary prediction table for the GLUE MRPC validation set produced by the sentence-transformers/all-distilroberta-v1 model. Each record corresponds to one validation example and stores a single field, prediction, where 1 indicates the sentence pair is predicted to be a paraphrase and 0 indicates not_paraphrase. Predictions are generated by computing the cosine similarity between the normalized embeddings of sentence1 and sentence2 and applying a fixed threshold of 0.72. This dataset serves as the model output layer in the workflow, linking each prediction back to the original MRPC example and the two sentence embedding datasets, and it is used downstream to compute accuracy, F1, error analysis, and the experiment summary."
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_mrpc_prediction_threshold_2", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "output_type": "binary prediction", "label_space": "0=not_paraphrase,1=paraphrase", "source": "glue/mrpc", "split": "validation", "size": "408", "input_type": "sentence pair", "model": "sentence-transformers/all-distilroberta-v1", "embedding_dim": "768", "similarity_metric": "cosine similarity", "threshold": "0.72", "prediction_column": "prediction", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_mrpc_prediction_threshold_2", cat, embedding, prop)


In [7]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.7230392156862745, 'f1': 0.8055077452667814}
                precision    recall  f1-score   support

not_paraphrase       0.58      0.47      0.52       129
    paraphrase       0.77      0.84      0.81       279

      accuracy                           0.72       408
     macro avg       0.68      0.66      0.66       408
  weighted avg       0.71      0.72      0.71       408



In [8]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9106887578964233
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.35538217425346375
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.7522245049476624
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will de

In [9]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

num_errors: 113
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.7522245049476624
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
score: 0.6570861339569092
true: 1 pred: 0
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
score: 0.8033972978591919
true: 0 pred: 1
idx: 11
sentence1: " Sanitation is poor ... there could be

In [10]:
vault.create_record_list("st_all_distilroberta_v1_cosine_threshold_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("st_all_distilroberta_v1_cosine_threshold_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "sentence_transformers_mrpc_prediction_threshold_2": [0, len(ds)]
                    })

summary

description = "Summary record list for the MRPC validation evaluation of a threshold-based paraphrase detection workflow using sentence-transformers/all-distilroberta-v1. This dataset contains experiment-level metrics computed by comparing predicted paraphrase labels from cosine similarity scores between sentence1 and sentence2 embeddings against the ground-truth labels in glue_mrpc_validation. Its structure is a record list with three fields: accuracy (float), f1 (float), and classification_report (string containing the full per-class evaluation report). In this notebook, it serves as the final aggregated result of the process, linking the full GLUE MRPC validation set and the generated prediction record list into a compact summary of model performance for the chosen cosine threshold."
embedding = get_embeddings(description)
vault.create_description("st_all_distilroberta_v1_cosine_threshold_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "benchmark": "GLUE", "subset": "MRPC", "split": "validation", "model": "sentence-transformers/all-distilroberta-v1", "representation": "sentence embeddings", "embedding_dimension": "768", "similarity_metric": "cosine similarity", "decision_rule": "threshold", "threshold": "0.72", "prediction_type": "binary classification", "labels": "not_paraphrase, paraphrase", "upstream_inputs": "glue_mrpc_validation, sentence-transformers-sentence-1-threshold_2, sentence-transformers-sentence-2-threshold_2", "metrics": "accuracy, f1, classification_report", "process_name": "st_all_distilroberta_v1_cosine_threshold_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_all_distilroberta_v1_cosine_threshold_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'sentence-transformers/all-distilroberta-v1',
 'device': 'mps',
 'threshold': 0.72,
 'num_examples': 408,
 'accuracy': 0.7230392156862745,
 'f1': 0.8055077452667814}

In [ ]:
description = "This notebook runs a paraphrase detection experiment on the GLUE MRPC validation set using the sentence-transformers/all-distilroberta-v1 model. It encodes each sentence in every sentence pair into normalized 768-dimensional embeddings, computes cosine similarity via the dot product, and applies a fixed threshold of 0.72 to predict whether the pair is a paraphrase. The workflow stores sentence embeddings, prediction records, evaluation outputs, and metadata in TableVault, linking all derived artifacts back to the original MRPC validation examples. It then evaluates performance with accuracy, F1, and a classification report, inspects sample predictions and errors, and saves a summary record for the full experiment." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("st_all_distilroberta_v1_cosine_threshold_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text pair classification", "method": "sentence embedding cosine similarity thresholding", "model": "sentence-transformers/all-distilroberta-v1", "embedding_model": "text-embedding-3-large", "dataset": "glue/mrpc validation", "framework": "sentence-transformers, PyTorch", "similarity_metric": "cosine similarity", "threshold": "0.72", "evaluation": "accuracy, f1-score, classification report", "prediction_artifact": "sentence_transformers_mrpc_prediction_threshold_2", "summary_artifact": "st_all_distilroberta_v1_cosine_threshold_mrpc_summary", "vector_dim": "768", "metadata_store": "TableVault with ArangoDB", "device": "mps_or_cpu", "input_type": "sentence pairs", "process_name": "st_all_distilroberta_v1_cosine_threshold_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_all_distilroberta_v1_cosine_threshold_mrpc", cat, embedding, prop)